In [ ]:
# Imports 
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#Data Loading & Setup

df = pd.read_csv('../data/processed/wine_combined.csv')

params = ['fixed acidity', 'volatile acidity', 'citric acid',
          'residual sugar', 'chlorides', 'free sulfur dioxide',
          'total sulfur dioxide', 'density', 'pH',
          'sulphates', 'alcohol']

# Shared colour mapping 
WINE_COLORS = {'Red': 'crimson', 'White': 'palegoldenrod'}

spec_limits = {
    'volatile acidity':    {'LSL': 0.08, 'USL': 1.2},   # OIV/US Federal limit
    'pH':                  {'LSL': 2.9,  'USL': 4.0},   # UC Davis / WineMakerMag
    'sulphates':           {'LSL': 0.25, 'USL': 1.5},   # Practical quality range
    'alcohol':             {'LSL': 8.5,  'USL': 15.0},  # EU Reg 1308/2013
    'free sulfur dioxide': {'LSL': 10.0, 'USL': 60.0},  # OIV Annex C
    'chlorides':           {'LSL': 0.005,'USL': 0.20},  # Practical quality range
}

print(f"Dataset loaded: {df.shape}")
print(f"Quality tiers: {df['quality_tier'].value_counts().to_dict()}")


In [ ]:
#Quality Driver Correlation Bar Chgart

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, wtype, color in zip(axes, ['Red', 'White'], ['crimson', 'palegoldenrod']):
    corr = df[df['wine_type'] == wtype][params + ['quality']].corr()['quality'].drop('quality').sort_values()
    
    # Color bars by direction — positive or negative correlation
    bar_colors = ['crimson' if x < 0 else 'palegoldenrod' for x in corr.values]
    
    ax.barh(corr.index, corr.values, color=bar_colors, edgecolor='white')
    ax.axvline(x=0, color='black', linewidth=0.8)
    ax.set_title(f'{wtype} Wine — Quality Drivers', fontsize=13)
    ax.set_xlabel('Correlation with Quality Score')
    ax.set_xlim(-0.6, 0.6)
    
    # Annotate each bar with its value
    for i, (val, name) in enumerate(zip(corr.values, corr.index)):
        ax.text(val + (0.02 if val >= 0 else -0.02), i, 
                f'{val:.2f}', va='center',
                ha='left' if val >= 0 else 'right',
                fontsize=9)

plt.suptitle('What Drives Wine Quality? — Correlation Analysis', 
             fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('../assets/quality_drivers.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Scatter Plots — Relationship of Top Chemical Drivers with Quality per Wine Type

top_drivers = {
    'Red':   ['alcohol', 'volatile acidity', 'sulphates'],
    'White': ['alcohol', 'density', 'chlorides']
}

for wtype, drivers in top_drivers.items():
    color = WINE_COLORS[wtype]
    subset = df[df['wine_type'] == wtype]

    for param in drivers:
        fig = px.scatter(subset, x=param, y='quality',
                         trendline='ols',
                         color_discrete_sequence=[color],
                         title=f'{wtype} Wine — {param.title()} vs Quality Score',
                         labels={param: param.title(), 'quality': 'Quality Score'},
                         opacity=0.4)

        fig.update_layout(paper_bgcolor='#E8E8E8', plot_bgcolor='white')
        fig.show()

In [ ]:
# DOES BEING OUT OF SPEC HURT QUALITY?

quality_impact = []

for param, spec in spec_limits.items():
    for wtype in ['Red', 'White']:
        subset = df[df['wine_type'] == wtype].copy()

        subset['oos_status'] = 'In-Spec'
        subset.loc[
            (subset[param] < spec['LSL']) | (subset[param] > spec['USL']),
            'oos_status'
        ] = 'Out-of-Spec'

        # Calculate average quality per group
        group_quality = subset.groupby('oos_status')['quality'].agg(
            ['mean', 'count']
        ).reset_index()

         # Only include if OOS samples exist
        if 'Out-of-Spec' not in group_quality['oos_status'].values:
            continue

        def group_val(status, col):
            return group_quality.loc[group_quality['oos_status'] == status, col].values[0]

        in_spec_q = group_val('In-Spec', 'mean')
        oos_q     = group_val('Out-of-Spec', 'mean')
        oos_count = group_val('Out-of-Spec', 'count')

        quality_impact.append({
            'Parameter':       param,
            'Wine Type':       wtype,
            'In-Spec Quality': round(in_spec_q, 3),
            'OOS Quality':     round(oos_q, 3),
            'Quality Delta':   round(oos_q - in_spec_q, 3),
            'OOS Count':       oos_count
        })

impact_df = pd.DataFrame(quality_impact)

print("Quality Impact of OOS Failures:")
print("=" * 70)
print(impact_df.to_string(index=False))


In [ ]:
# QUALITY SCORE DISTRIBUTION BY OOS STATUS

fig, axes = plt.subplots(len(spec_limits), 2,
                         figsize=(14, len(spec_limits) * 3))

for row, (param, spec) in enumerate(spec_limits.items()):
    for col, wtype in enumerate(['Red', 'White']):
        ax = axes[row, col]
        subset = df[df['wine_type'] == wtype].copy()

         # Tag OOS status
        subset['oos_status'] = 'In-Spec'
        subset.loc[
            (subset[param] < spec['LSL']) | (subset[param] > spec['USL']),
            'oos_status'
        ] = 'Out-of-Spec'

         # Only plot if OOS samples exist
        oos_count = (subset['oos_status'] == 'Out-of-Spec').sum()

        if oos_count == 0:
            ax.set_visible(False)
            continue

        subset.boxplot(
            column='quality',
            by='oos_status',
            ax=ax,
            boxprops=dict(color='steelblue'),
            medianprops=dict(color='crimson', linewidth=2),
            whiskerprops=dict(color='steelblue'),
            capprops=dict(color='steelblue'),
            flierprops=dict(marker='o', alpha=0.3,
                            markerfacecolor='grey', markersize=3)
        )
        ax.set_title(f'{param.title()} — {wtype} Wine\n(OOS n={oos_count})', fontsize=9)
        ax.set_xlabel('')
        ax.set_ylabel('Quality Score', fontsize=8)
        ax.set_facecolor('white')

plt.suptitle('Quality Score: In-Spec vs Out-of-Spec Samples', fontsize=14, y=1.01)
plt.tight_layout()
fig.patch.set_facecolor('#E8E8E8')
plt.savefig('../assets/quality_oos_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ROOT CAUSE SUMMARY TABLE

summary_df = pd.read_csv('../data/processed/spc_summary.csv')
print("✅ SPC summary table loaded👇")
print(summary_df.to_string(index=False))

# Merge SPC summary with quality impact 
root_causes = {
    'volatile acidity':    'Acetic acid bacteria — microbial spoilage',
    'pH':                  'Acid imbalance — grape variety or fermentation',
    'sulphates':           'Inconsistent SO₂ addition protocol',
    'alcohol':             'Incomplete fermentation — yeast health issue',
    'free sulfur dioxide': 'Inconsistent SO₂ addition — preservation risk',
    'chlorides':           'Terroir variation — water source or soil salinity',
}

def quality_risk(delta):
    if delta < -0.5: return '🚨 High'
    if delta < 0:    return '⚠️ Medium'
    return '✅ Low'

root_cause_rows = []

for param, spec in spec_limits.items():
    for wtype in ['Red', 'White']:
        # Get SPC verdict from Phase 3 summary df
        spc_row = summary_df[
            (summary_df['Parameter'] == param) &
            (summary_df['Wine Type'] == wtype)
        ].iloc[0]

        # Get quality impact from Phase 4
        impact_row = impact_df[
            (impact_df['Parameter'] == param) &
            (impact_df['Wine Type'] == wtype)
        ]

        # Quality delta
        if len(impact_row) > 0:
            delta     = impact_row['Quality Delta'].values[0]
            oos_count = impact_row['OOS Count'].values[0]
        else:
            delta, oos_count = 0.0, 0


        root_cause_rows.append({
            'Wine Type':     wtype,
            'Parameter':     param,
            'SPC Verdict':   spc_row['Verdict'],
            'Cpk':           spc_row['Cpk'],
            'OOS %':         spc_row['OOS %'],
            'OOS Count':     oos_count,
            'Quality Delta': delta,
            'Quality Risk':  quality_risk(delta),
            'Root Cause':    root_causes[param]
        })

rca_df = pd.DataFrame(root_cause_rows)

plain_fill = ['#f9f9f9'] * len(rca_df)

fig = go.Figure(data=[go.Table(
    columnwidth=[80, 130, 100, 60, 60, 70, 90, 90, 200],
    header=dict(
        values=['Wine Type', 'Parameter', 'SPC Verdict',
                'Cpk', 'OOS %', 'OOS Count',
                'Quality Delta', 'Quality Risk', 'Root Cause'],
        fill_color='#404040',
        font=dict(color='white', size=11),
        align='center',
        height=35
    ),
    cells=dict(
        values=[rca_df[col] for col in rca_df.columns],
        fill_color=[
            *[plain_fill] * 2,
            ['#f8d7da' if v == '🚨 Critical'
             else '#fff3cd' if v == '⚠️ Review'
             else '#d4edda' for v in rca_df['SPC Verdict']],
            *[plain_fill] * 4,
            ['#f8d7da' if v == '🚨 High'
             else '#fff3cd' if v == '⚠️ Medium'
             else '#d4edda' for v in rca_df['Quality Risk']],
            plain_fill,
        ],
        font=dict(size=10),
        align='center',
        height=30
    )
)])

fig.update_layout(
    title='Root Cause Analysis — SPC Verdict vs Quality Impact',
    paper_bgcolor='#E8E8E8',
    height=550
)

fig.show()

print("\nRoot Cause Analysis Summary:")
print("=" * 90)
print(rca_df.to_string(index=False))
